# Notebook 04 — Evaluation

Reads the runs recorded by notebook 03 and turns them into the figures and the table the thesis needs.

Nothing here trains or decides anything: every choice was made on training and validation, and the test split was touched once per run, at the end of notebook 03. This notebook only reads what those runs wrote.

Two things it produces that a metric alone cannot give:

- **error maps**, which separate predicting too much from finding too little — two failures with the same IoU and very different consequences for pruning
- **the per-class comparison**, because a model can lead on mean IoU while losing the two thin classes the task is about

# 0. Environment Setup

Unlike notebook 03 this one needs no GPU: it reads checkpoints and runs a handful of forward passes. It works locally, except that predicting a split needs the online pipeline, and therefore `albumentations`.

In [ ]:
# =====================================================
# 0.1 - BOOTSTRAP: DRIVE MOUNT AND MODULE PATH
# =====================================================
import importlib
import sys
from pathlib import Path

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
else:
    PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "src" / "config.py").exists():
    raise FileNotFoundError(f"config.py not found under: {PROJECT_ROOT / 'src'}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config


def reload_project_modules() -> None:
    """Reload the modules in src/ after editing them."""
    for module_name in (
        "config", "data_preparation", "exploration",
        "dataset", "losses", "metrics", "models", "training", "evaluation",
    ):
        if module_name in sys.modules:
            importlib.reload(sys.modules[module_name])
            print(f"  reloaded: {module_name}")


# Figures for the thesis go here rather than into a run folder: several of them
# compare runs and belong to none.
FIGURES_DIR = config.RESULTS_DIR / "figures" / "evaluation"

config.print_environment_info()
config.set_seed()

# 1. Choose a Run

Lists the runs that hold a usable checkpoint and rebuilds one of them.

The architecture and width come from the run's own `config.json`, not from the live `config.py`: the point of comparing runs is that they differ, so reading the current configuration would rebuild the wrong network.

In [ ]:
# =====================================================
# 1.1 - WHAT IS AVAILABLE
# =====================================================
from evaluation import list_runs

RUNS = list_runs()

for run_id in RUNS:
    print(f"  {run_id}")

if not RUNS:
    raise RuntimeError("No finished run found. Run notebook 03 first.")

In [ ]:
# =====================================================
# 1.2 - LOAD ONE
# =====================================================
from evaluation import load_run

RUN_ID = RUNS[0]          # most recent; set explicitly to inspect another

MODEL, RUN_CONFIG = load_run(RUN_ID)

print(f"Run:           {RUN_ID}")
print(f"Architecture:  {RUN_CONFIG['model']['architecture']} "
      f"(base {RUN_CONFIG['model']['base_channels']}, "
      f"{RUN_CONFIG['model'].get('parameters', 0):,} parameters)")
print(f"Loss:          {RUN_CONFIG['loss']['type']}")
print(f"Skeleton loss: {RUN_CONFIG['skeleton_loss']['applied']}")
print(f"Commit:        {RUN_CONFIG.get('git_commit')}")

# 2. Predictions on the Test Split

Runs the loaded model over the test split and scores it.

The dataloader must not shuffle, or the predictions stop lining up with the file stems and every figure below would show the wrong pair. `create_dataloaders` only shuffles the training split, so this holds by construction — but it is the assumption that makes the rest valid.

In [ ]:
# =====================================================
# 2.1 - PREDICT AND SCORE
# =====================================================
from dataset import build_transforms, create_dataloaders
from data_preparation import get_splits_hash
from evaluation import evaluate_predictions, predict_split

SPLITS_HASH = get_splits_hash()
TRANSFORMS = build_transforms(splits_hash=SPLITS_HASH)
DATALOADERS = create_dataloaders(transforms=TRANSFORMS, verbose=False)

PREDICTIONS, TARGETS = predict_split(MODEL, DATALOADERS["test"])
STEMS = DATALOADERS["test"].dataset.stems

METRICS = evaluate_predictions(PREDICTIONS, TARGETS)

In [ ]:
# =====================================================
# 2.2 - CENTRELINE DICE
# =====================================================
# Reported apart from the overlap scores because it answers a different
# question: whether the structure was found, not how precisely its edge was
# placed. A low IoU next to a high clDice means the topology is right and the
# boundary is not — and on thin structures part of that gap is the annotation,
# not the model.
from metrics import print_cldice_report

print_cldice_report({
    "cldice_per_class": METRICS["cldice_per_class"],
    "precision_per_class": METRICS["cldice_precision_per_class"],
    "recall_per_class": METRICS["cldice_recall_per_class"],
    "mean_cldice": METRICS["mean_cldice"],
})

# 3. What the Model Gets Wrong

Four panels per sample: the intensity image, the annotation, the prediction, and an error map that colours the mistakes by kind.

The error map is what the numbers cannot say. Red is a structure predicted where there is none, blue is one present and not found, yellow is one found and called the wrong class. On this dataset the expected signature of over-segmentation is a **red halo tracing the true structures** rather than red blobs away from them — inflation of the boundary, not invention of new canes.

Output: `results/figures/evaluation/`

In [ ]:
# =====================================================
# 3.1 - A FEW SAMPLES
# =====================================================
from evaluation import show_predictions

show_predictions(
    PREDICTIONS, TARGETS, STEMS,
    n_samples=3,
    save_path=FIGURES_DIR / f"{RUN_ID}_predictions.png",
)

In [ ]:
# =====================================================
# 3.2 - THE WORST CASES
# =====================================================
# Averages hide the systematic weaknesses; the failures are where they show,
# and a thesis figure should be honest about them.
from evaluation import worst_samples

WORST = worst_samples(PREDICTIONS, TARGETS, class_index=config.CLASS_NAMES.index("Cane"), count=3)
print(f"Worst on Cane: {[STEMS[i] for i in WORST]}")

show_predictions(
    PREDICTIONS, TARGETS, STEMS,
    indices=WORST,
    save_path=FIGURES_DIR / f"{RUN_ID}_worst.png",
)

# 4. Comparison Across Runs

Reads `results/experiments.csv` and puts the runs side by side.

Runs measured on another partition are hidden: metrics from different splits are not comparable, and mixing them into one table invites exactly the wrong conclusion. The `splits_hash` column is what makes that check possible.

The ordering column has to be declared rather than assumed. Sorting by mean IoU is one choice among several, and for pruning it is not obviously the right one — a missed cane costs more than an inflated one.

In [ ]:
# =====================================================
# 4.1 - THE TABLE
# =====================================================
from evaluation import comparison_table, load_results

RESULTS = load_results()
comparison_table(RESULTS, sort_by="test_mean_iou_no_bg")

In [ ]:
# =====================================================
# 4.2 - THE SAME TABLE, READ THE OTHER WAY
# =====================================================
# Sorted by clDice instead. If the order changes, the two metrics disagree
# about which run is best — which is information, not a problem: it means the
# runs differ in whether they get the topology or the boundary right.
comparison_table(RESULTS, sort_by="test_mean_cldice")

# 5. Per-Class Comparison

The mean over classes hides the ones that matter. `Background` is 84 % of the pixels and easy; `Node` is 0.27 % and hard. A run can lead on the mean while losing both thin classes.

If the skeleton recall loss behaves as section 7.6 of the notes predicts — rewarding recall without penalising false positives — it should show here as **precision falling on `Cane` and `Cordon` while recall holds or rises**. Plotting the two next to each other is what makes that visible.

Output: `results/figures/evaluation/`

In [ ]:
# =====================================================
# 5.1 - IOU AND CLDICE PER CLASS
# =====================================================
from evaluation import plot_class_comparison

plot_class_comparison(RESULTS, metric="iou", save_path=FIGURES_DIR / "class_iou.png")
plot_class_comparison(RESULTS, metric="cldice", save_path=FIGURES_DIR / "class_cldice.png")

In [ ]:
# =====================================================
# 5.2 - PRECISION AGAINST RECALL, PER CLASS
# =====================================================
# The pair to read for the skeleton loss ablation.
plot_class_comparison(RESULTS, metric="precision", save_path=FIGURES_DIR / "class_precision.png")
plot_class_comparison(RESULTS, metric="recall", save_path=FIGURES_DIR / "class_recall.png")